
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 3 - Flight Data ETL with the DataFrame API

This demonstration will walk through common ETL operations using the Flights dataset. We'll cover data loading, cleaning, transformation, and analysis using the DataFrame API.

### Objectives
- Implement common ETL operations using Spark DataFrames
- Handle data cleaning and type conversion
- Create derived features through transformations
- Use different column reference methods
- Work with User Defined Functions (UDFs)

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>

```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run ./Includes/Classroom-Setup-Common

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Course Catalog:,
Your Schema:,


DataFrame[]

View your default catalog and schema.

In [0]:
%sql
SELECT current_catalog(), current_schema()

current_catalog(),current_schema()
dbacademy,labuser10760585_1751060277


### Flight Data Processing Requirements

#### Source Data
Dataset Location: `dbacademy_airline.v01.flights_small`(flight information dataset)

#### Target
Table name: flight_data

Schema:

| Column Name | Data Type | Description |
|-------------|-----------|-------------|
| FlightDateTime | datetime | Datetime of the flight (derived from the Year, Month, DayofMonth, DepTime fields in the source data) |
| FlightNum | integer | Flight number |
| ElapsedTimeDiff | integer | Difference between scheduled elapsed time and actual elapsed time for the flight, derived from the ActualElapsedTime and CRSElapsedTime fields in the source data |
| ArrDelayCategory | string | Categories include "On Time", "Slight Delay", "Moderate Delay" and "Severe Delay" based upon the value of the ArrDelay in the source data |

## B. Data Loading and Inspection

First, let's load and inspect the flight data.

In [0]:
# Read the flights data
flights_df = spark.read.table("dbacademy_airline.v01.flights_small")

In [0]:
# Print the schema
flights_df.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true)
 |-- Car

In [0]:
# Visually inspect a subset of the data
display(flights_df.limit(10))

Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,IsArrDelayed,IsDepDelayed
1998,1,17,6,715,715,840,840,WN,76,N520,145,145,137,0,0,AUS,PHX,872,3,5,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,18,7,715,715,830,840,WN,76,N302,135,145,127,-10,0,AUS,PHX,872,2,6,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,19,1,715,715,831,840,WN,76,N315,136,145,128,-9,0,AUS,PHX,872,3,5,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,20,2,715,715,836,840,WN,76,N372,141,145,126,-4,0,AUS,PHX,872,3,12,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,21,3,715,715,836,840,WN,76,N367,141,145,132,-4,0,AUS,PHX,872,3,6,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,22,4,725,715,847,840,WN,76,N318,142,145,131,7,10,AUS,PHX,872,4,7,0,NA,0,NA,NA,NA,NA,NA,YES,YES
1998,1,23,5,715,715,839,840,WN,76,N694,144,145,121,-1,0,AUS,PHX,872,4,19,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,24,6,715,715,830,840,WN,76,N373,135,145,126,-10,0,AUS,PHX,872,4,5,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,25,7,715,715,835,840,WN,76,N302,140,145,133,-5,0,AUS,PHX,872,2,5,0,NA,0,NA,NA,NA,NA,NA,NO,NO
1998,1,26,1,715,715,840,840,WN,76,N307,145,145,133,0,0,AUS,PHX,872,3,9,0,NA,0,NA,NA,NA,NA,NA,NO,NO


In [0]:
# Let's remove columns we dont need, remember "filter early, filter often"
flights_required_cols_df = flights_df.select(
    "Year",
    "Month",
    "DayofMonth",
    "DepTime",
    "FlightNum",
    "ActualElapsedTime",
    "CRSElapsedTime",
    "ArrDelay")

# Alternatively we could have used the drop() method to remove the columns we didnt want...

In [0]:
# Get a count of the source data records
initial_count = flights_required_cols_df.count()

print(f"Source data has {initial_count} records")

Source data has 10602522 records


In [0]:
# Let's examine the data for invalid values, these can include nulls or invalid values for string columns "ArrDelay", "ActualElapsedTime", "DepTime" which we intend on performing mathematical operations on, we can use the Spark SQL COUNT_IF function to perform the analysis

# Register the DataFrame as a temporary SQL table with cast columns
flights_required_cols_df \
    .selectExpr(
        "Year",
        "Month",
        "DayofMonth",
        "CAST(DepTime AS INT) AS DepTime",
        "FlightNum",
        "CAST(ActualElapsedTime AS INT) AS ActualElapsedTime",
        "CRSElapsedTime",
        "CAST(ArrDelay AS INT) AS ArrDelay"
    ) \
    .createOrReplaceTempView("flights_temp")

In [0]:
# Use Spark SQL to count null values
invalid_counts_sql = spark.sql("""
SELECT 
    COUNT_IF(Year IS NULL) AS Null_Year_Count,
    COUNT_IF(Month IS NULL) AS Null_Month_Count,
    COUNT_IF(DayofMonth IS NULL) AS Null_DayOfMonth_Count,
    COUNT_IF(DepTime IS NULL) AS Null_DepTime_Count,
    COUNT_IF(FlightNum IS NULL) AS Null_FlightNum_Count,
    COUNT_IF(ActualElapsedTime IS NULL) AS Null_ActualElapsedTime_Count,
    COUNT_IF(CRSElapsedTime IS NULL) AS Null_CRSElapsedTime_Count,
    COUNT_IF(ArrDelay IS NULL) AS Null_ArrDelay_Count
FROM flights_temp
""")

display(invalid_counts_sql)

Null_Year_Count,Null_Month_Count,Null_DayOfMonth_Count,Null_DepTime_Count,Null_FlightNum_Count,Null_ActualElapsedTime_Count,Null_CRSElapsedTime_Count,Null_ArrDelay_Count
0,0,0,292961,0,319093,11375,319093


### 1. Comparing Spark SQL to DataFrame API Operations
Spark SQL DataFrame queries and their equivalent operations in the DataFrame API are evaluated to the same physical plans, let's prove this.

In [0]:
# this is the equivalent of the preceding Spark SQL query using the DataFrame API
from pyspark.sql.functions import col, sum, when

# Make sure to work with the same temporary view that the SQL is using
flights_temp_df = spark.table("flights_temp")

# Use DataFrame API to count null values
invalid_counts_df = flights_temp_df.select(
    sum(when(col("Year").isNull(), 1).otherwise(0)).alias("Null_Year_Count"),
    sum(when(col("Month").isNull(), 1).otherwise(0)).alias("Null_Month_Count"),
    sum(when(col("DayofMonth").isNull(), 1).otherwise(0)).alias("Null_DayOfMonth_Count"),
    sum(when(col("DepTime").isNull(), 1).otherwise(0)).alias("Null_DepTime_Count"),
    sum(when(col("FlightNum").isNull(), 1).otherwise(0)).alias("Null_FlightNum_Count"),
    sum(when(col("ActualElapsedTime").isNull(), 1).otherwise(0)).alias("Null_ActualElapsedTime_Count"),
    sum(when(col("CRSElapsedTime").isNull(), 1).otherwise(0)).alias("Null_CRSElapsedTime_Count"),
    sum(when(col("ArrDelay").isNull(), 1).otherwise(0)).alias("Null_ArrDelay_Count")
)

display(invalid_counts_df)

Null_Year_Count,Null_Month_Count,Null_DayOfMonth_Count,Null_DepTime_Count,Null_FlightNum_Count,Null_ActualElapsedTime_Count,Null_CRSElapsedTime_Count,Null_ArrDelay_Count
0,0,0,292961,0,319093,11375,319093


In [0]:
# Get the explain plans for the SQL and DF versions of our query
sql_plan = invalid_counts_sql.explain() #Getting SQL Plan Details

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   HashAggregate(keys=[], functions=[finalmerge_count(merge count#3430L) AS count(if (isnotnull(Year#2871)) null else isnull(Year#2871))#3211L, finalmerge_count(merge count#3432L) AS count(if (isnotnull(Month#2872)) null else isnull(Month#2872))#3212L, finalmerge_count(merge count#3434L) AS count(if (isnotnull(DayofMonth#2873)) null else isnull(DayofMonth#2873))#3213L, finalmerge_count(merge count#3436L) AS count(if (isnotnull(DepTime#3192)) null else isnull(DepTime#3192))#3214L, finalmerge_count(merge count#3438L) AS count(if (isnotnull(FlightNum#2880)) null else isnull(FlightNum#2880))#3215L, finalmerge_count(merge count#3440L) AS count(if (isnotnull(ActualElapsedTime#3193)) null else isnull(ActualElapsedTime#3193))#3216L, finalmerge_count(merge count#3442L) AS count(if (isnotnull(CRSElapsedTime#2883)) null else isnull(CRSElapsedTime#2883))#3217L, finalmerge_count(merge count#3444L) AS count(if (isnotnull(A

In [0]:
df_plan = invalid_counts_df.explain() # Getting DF Plan Details

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   HashAggregate(keys=[], functions=[finalmerge_sum(merge sum#3353L) AS sum(CASE WHEN isnull(Year#2871) THEN 1 ELSE 0 END)#3328L, finalmerge_sum(merge sum#3355L) AS sum(CASE WHEN isnull(Month#2872) THEN 1 ELSE 0 END)#3329L, finalmerge_sum(merge sum#3357L) AS sum(CASE WHEN isnull(DayofMonth#2873) THEN 1 ELSE 0 END)#3330L, finalmerge_sum(merge sum#3359L) AS sum(CASE WHEN isnull(DepTime#3192) THEN 1 ELSE 0 END)#3331L, finalmerge_sum(merge sum#3361L) AS sum(CASE WHEN isnull(FlightNum#2880) THEN 1 ELSE 0 END)#3332L, finalmerge_sum(merge sum#3363L) AS sum(CASE WHEN isnull(ActualElapsedTime#3193) THEN 1 ELSE 0 END)#3333L, finalmerge_sum(merge sum#3365L) AS sum(CASE WHEN isnull(CRSElapsedTime#2883) THEN 1 ELSE 0 END)#3334L, finalmerge_sum(merge sum#3367L) AS sum(CASE WHEN isnull(ArrDelay#3194) THEN 1 ELSE 0 END)#3335L])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1035]
      +- HashAggregate(keys=[]

In [0]:
# Show that the two approaches evaluate to the same physical plan
sql_plan == df_plan

True

### 2. Using the Databricks AI Assistant
The Databricks AI Assistant feature can be used to generate code or to visualize metrics from DataFrames, from the code cell below click on the __generate__ link and enter:

```generate a bar chart showing nulls for each column in the flights_temp_df dataframe```

**NOTE:** Click on AI assistance toggle button and Enter the given prompt.

In [0]:
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.functions import *

# Add your AI generated code from above cell

## C. Data Cleaning

The flights data contains some invalid and missing values, lets find them and clean them (in this case we will drop them)

In [0]:
# To drop rows where any specified columns are null, we can use the na.drop DataFrame method
non_null_flights_df = flights_required_cols_df.na.drop(
    how='any',
    subset=['CRSElapsedTime']
)

In [0]:
from pyspark.sql.functions import col

# Let's remove rows with invalid values for "ArrDelay", "ActualElapsedTime" and "DepTime" columns
flights_with_valid_data_df = non_null_flights_df.filter(
    col("ArrDelay").cast("integer").isNotNull() & 
    col("ActualElapsedTime").cast("integer").isNotNull() &
    col("DepTime").cast("integer").isNotNull()
)

In [0]:
# Now that we know "ArrDelay" and "ActualElapsedTime" contain integer values only, lets cast them from strings to integers (replacing the existing columns)
clean_flights_df = flights_with_valid_data_df \
    .withColumn("ArrDelay", col("ArrDelay").cast("integer")) \
    .withColumn("ActualElapsedTime", col("ActualElapsedTime").cast("integer"))

clean_flights_df.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- ActualElapsedTime: integer (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)



## D. Data Enrichment

Now let's create a useful derived column to categorize delays.

In [0]:
# Let's start by deriving the "FlightDateTime" column from the "Year", "Month", "DayofMonth", "DepTime" columns, then drop the constituent columns
from pyspark.sql.functions import col, make_timestamp_ntz, lpad, substr, lit

flights_with_datetime_df = clean_flights_df.withColumn(
    "FlightDateTime",
    make_timestamp_ntz(
        col("Year"),
        col("Month"),
        col("DayofMonth"),
        substr(lpad(col("DepTime"), 4, "0"), lit(1), lit(2)).cast("integer"),
        substr(lpad(col("DepTime"), 4, "0"), lit(3), lit(2)).cast("integer"),
        lit(0)
    )
).drop("Year", "Month", "DayofMonth", "DepTime")

# Show the result
display(flights_with_datetime_df.limit(10))

FlightNum,ActualElapsedTime,CRSElapsedTime,ArrDelay,FlightDateTime
996,123,129,25,1999-04-27T15:31:00
996,125,129,0,1999-04-28T15:04:00
996,134,129,29,1999-04-29T15:24:00
996,142,129,74,1999-04-30T16:01:00
1258,132,124,19,1999-04-01T13:41:00
1258,138,124,32,1999-04-02T13:48:00
1258,120,124,-2,1999-04-03T13:32:00
1420,121,145,-15,1999-04-04T17:39:00
1420,129,145,-10,1999-04-05T17:36:00
1420,112,145,-25,1999-04-06T17:38:00


In [0]:
# Lets derive the "ElapsedTimeDiff" column from the "ActualElapsedTime" and "CRSElapsedTime" columns

from pyspark.sql.functions import col

flights_with_elapsed_time_diff_df = flights_with_datetime_df.withColumn(
    "ElapsedTimeDiff", col("ActualElapsedTime") - col("CRSElapsedTime")
    ).drop("ActualElapsedTime", "CRSElapsedTime")

display(flights_with_elapsed_time_diff_df.limit(10))

FlightNum,ArrDelay,FlightDateTime,ElapsedTimeDiff
996,25,1999-04-27T15:31:00,-6
996,0,1999-04-28T15:04:00,-4
996,29,1999-04-29T15:24:00,5
996,74,1999-04-30T16:01:00,13
1258,19,1999-04-01T13:41:00,8
1258,32,1999-04-02T13:48:00,14
1258,-2,1999-04-03T13:32:00,-4
1420,-15,1999-04-04T17:39:00,-24
1420,-10,1999-04-05T17:36:00,-16
1420,-25,1999-04-06T17:38:00,-33


In [0]:
# Now lets categorize the "ArrDelay" column into categories: "On Time", "Slight Delay", "Moderate Delay", "Severe Delay"

from pyspark.sql.functions import when

enriched_flights_df = flights_with_elapsed_time_diff_df \
    .withColumn("delay_category", when(col("ArrDelay") <= 0, "On Time")
        .when(col("ArrDelay") <= 15, "Slight Delay")
        .when(col("ArrDelay") <= 60, "Moderate Delay")
        .otherwise("Severe Delay")) \
       .drop("ArrDelay")
    
display(enriched_flights_df.limit(10))

FlightNum,FlightDateTime,ElapsedTimeDiff,delay_category
996,1999-04-27T15:31:00,-6,Moderate Delay
996,1999-04-28T15:04:00,-4,On Time
996,1999-04-29T15:24:00,5,Moderate Delay
996,1999-04-30T16:01:00,13,Severe Delay
1258,1999-04-01T13:41:00,8,Moderate Delay
1258,1999-04-02T13:48:00,14,Moderate Delay
1258,1999-04-03T13:32:00,-4,On Time
1420,1999-04-04T17:39:00,-24,On Time
1420,1999-04-05T17:36:00,-16,On Time
1420,1999-04-06T17:38:00,-33,On Time


## E. Analyze Delays

Let's analyze our delay categories using various column referencing approaches.

In [0]:
# Direct reference to list 100 random records
display(enriched_flights_df.select("FlightNum", "delay_category").limit(100))

FlightNum,delay_category
996,Moderate Delay
996,On Time
996,Moderate Delay
996,Severe Delay
1258,Moderate Delay
1258,Moderate Delay
1258,On Time
1420,On Time
1420,On Time
1420,On Time


In [0]:
# Column object
display(enriched_flights_df.select(col("FlightNum").alias("carrier_code"), col("delay_category")).limit(100))

carrier_code,delay_category
996,Moderate Delay
996,On Time
996,Moderate Delay
996,Severe Delay
1258,Moderate Delay
1258,Moderate Delay
1258,On Time
1420,On Time
1420,On Time
1420,On Time


In [0]:
# String expressions
display(enriched_flights_df.selectExpr("FlightNum", "ElapsedTimeDiff", "ElapsedTimeDiff > 0 as LongerThanScheduled"))

FlightNum,ElapsedTimeDiff,LongerThanScheduled
996,-6,false
996,-4,false
996,5,true
996,13,true
1258,8,true
1258,14,true
1258,-4,false
1420,-24,false
1420,-16,false
1420,-33,false


## F. Working with UDFs

Let's use a vectorized UDF to calculate the z-score (standard deviations from the mean) for delays for each flight

In [0]:
from pyspark.sql.functions import pandas_udf

# Pandas UDF (vectorized)
@pandas_udf("double")
def normalized_diff(diff_series):
    return (diff_series - diff_series.mean()) / diff_series.std()

# Apply both UDFs
udf_example = enriched_flights_df \
    .withColumn("diff_normalized", normalized_diff("ElapsedTimeDiff"))

display(udf_example)

# Note: In practice, prefer built-in functions over UDFs when possible

FlightNum,FlightDateTime,ElapsedTimeDiff,delay_category,diff_normalized
996,1999-04-27T15:31:00,-6,Moderate Delay,-0.4742886427851955
996,1999-04-28T15:04:00,-4,On Time,-0.3420200715360473
996,1999-04-29T15:24:00,5,Moderate Delay,0.2531884990851194
996,1999-04-30T16:01:00,13,Severe Delay,0.7822627840817121
1258,1999-04-01T13:41:00,8,Moderate Delay,0.4515913559588417
1258,1999-04-02T13:48:00,14,Moderate Delay,0.8483970697062861
1258,1999-04-03T13:32:00,-4,On Time,-0.3420200715360473
1420,1999-04-04T17:39:00,-24,On Time,-1.664705784027529
1420,1999-04-05T17:36:00,-16,On Time,-1.1356314990309364
1420,1999-04-06T17:38:00,-33,On Time,-2.2599143546486955


## G. Putting it altogether

Let's put this together in a chained operation to manipulate data from a source system and save it to a new target (overwriting any existing data)

In [0]:
%sql
-- Drop the target table in case it exists already
DROP TABLE IF EXISTS cleaned_and_enriched_flights;

In [0]:
from pyspark.sql.functions import col, make_timestamp_ntz, lpad, substr, lit, when, pandas_udf
# or more simply...
from pyspark.sql.functions import *

@pandas_udf("double")
def normalized_diff(diff_series):
    return (diff_series - diff_series.mean()) / diff_series.std()

(spark.read.table("dbacademy_airline.v01.flights_small")
    .selectExpr(
        "Year",
        "Month",
        "DayofMonth",
        "CAST(DepTime AS INT) AS DepTime",
        "FlightNum",
        "CAST(ActualElapsedTime AS INT) AS ActualElapsedTime",
        "CRSElapsedTime",
        "CAST(ArrDelay AS INT) AS ArrDelay"
    )
    .na.drop()
    .withColumn(
        "FlightDateTime",
        make_timestamp_ntz(
            col("Year"),
            col("Month"),
            col("DayofMonth"),
            substr(lpad(col("DepTime"), 4, "0"), lit(1), lit(2)).cast("integer"),
            substr(lpad(col("DepTime"), 4, "0"), lit(3), lit(2)).cast("integer"),
            lit(0)
        )
    )
    .drop("Year", "Month", "DayofMonth", "DepTime")
    .withColumn(
        "ElapsedTimeDiff", col("ActualElapsedTime") - col("CRSElapsedTime")
        )
    .drop("ActualElapsedTime", "CRSElapsedTime")
    .withColumn("delay_category", when(col("ArrDelay") <= 0, "On Time")
        .when(col("ArrDelay") <= 15, "Slight Delay")
        .when(col("ArrDelay") <= 60, "Moderate Delay")
        .otherwise("Severe Delay")) \
    .drop("ArrDelay")
    .withColumn("diff_normalized", normalized_diff("ElapsedTimeDiff"))
    # Write optimized
    .write
    .mode("overwrite")
    .saveAsTable("cleaned_and_enriched_flights"))

In [0]:
%sql
SELECT * FROM cleaned_and_enriched_flights;

FlightNum,FlightDateTime,ElapsedTimeDiff,delay_category,diff_normalized
76,1998-01-17T07:15:00,0,On Time,0.1989699449199636
76,1998-01-18T07:15:00,-10,On Time,-0.6426252878013103
76,1998-01-19T07:15:00,-9,On Time,-0.558465764529183
76,1998-01-20T07:15:00,-4,On Time,-0.137668148168546
76,1998-01-21T07:15:00,-4,On Time,-0.137668148168546
76,1998-01-22T07:25:00,-3,Slight Delay,-0.05350862489641861
76,1998-01-23T07:15:00,-1,On Time,0.11481042164783618
76,1998-01-24T07:15:00,-10,On Time,-0.6426252878013103
76,1998-01-25T07:15:00,-5,On Time,-0.22182767144067342
76,1998-01-26T07:15:00,0,On Time,0.1989699449199636


## Key Takeaways

1. **Data Cleaning Best Practices**:
   - Validate and clean data types early
   - Handle missing values appropriately
   - Document cleaning assumptions

2. **Data Enrichment**:
   - Create meaningful derived columns
   - Consider business requirements
   - Use functions (built-in or user defined to enrich datasets)



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
